# Model Training on Reduced Embeddings (Leakage-Free CV)
*Evaluation of classification models trained on low-dimensional embeddings of Heart Failure Prediction Dataset*  

**Important**: Previously saved embeddings (from `dim_red.ipynb`) were computed on the full dataset. They are suitable for **EDA/visualisation**, but **must not** cannot be used for unbiased model evaluation. Here, dimensionality reduction is fitted **inside each cross-validation fold** (train-only) and then applied to the corresponding test split.

---

## Table of Contents
1. [Setup](#setup)  
    1.2 [Imports and Reproducibility](#imports-and-reproducibility)  
    1.2 [Data Loading](#data-loading)  

2. [Baseline (full feature space)](#baseline)
   
3. [Model Training on Dimensionality-Reduced Feature Spaces](#model-on-dim-red)
   
4. [Conclusions](#conclusions)  

<a id="setup"></a>
## Setup
---
<a id="imports-and-reproducibility"></a>
### Imports and Reproducibility

In [2]:
import os, joblib, sys, pandas as pd
sys.path.append(os.path.abspath(os.path.join('..')))

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, f1_score
from sklearn.base import clone

SEED = 42

<a i="data-loading"></a>
### Data loading

In [3]:
df = joblib.load("../outputs/eda/df_preprocessed_balanced.pkl")
df_copy = df.copy()

X = df_copy.iloc[:, :-1]
y = df_copy.iloc[:, -1]

TARGET_COL = 'DEATH_EVENT'

print("X shape:", X.shape, "\n")
print("y distribution:")
print(pd.Series(y).value_counts(normalize=True).rename("share"))

X shape: (308, 12) 

y distribution:
DEATH_EVENT
0    0.659091
1    0.340909
Name: share, dtype: float64


<a id="baseline"></a>
## Baseline (full feature space)
---
Before comparing embeddings, a baseline using the full preprocessed feature space is established.

In [4]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
SCORING = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "f1": make_scorer(f1_score),
    "roc_auc": "roc_auc",
}

Previously optimised estimator configurations were loaded from disk. Models were cloned (fitted state removed) and refit from scratch inside cross-validation, so no pretrained parameters were reused.

In [5]:
models = {
    "RandomForest": clone(joblib.load("../outputs/models/best/RandomForest.pkl")),
    "AdaBoost": clone(joblib.load("../outputs/models/best/AdaBoost.pkl")),
    "XGBoost": clone(joblib.load("../outputs/models/best/XGBoost.pkl")),
    "Naive Bayes": clone(joblib.load("../outputs/models/best/NaiveBayes.pkl")),
    "SVM": clone(joblib.load("../outputs/models/best/SVM.pkl")), 
    "Logistic Regression": clone(joblib.load("../outputs/models/best/LogisticRegression.pkl")),
    "Stacking": clone(joblib.load("../outputs/models/best/STACKING.pkl")),
    "Voting": clone(joblib.load("../outputs/models/best/VOTING.pkl"))
}

In [6]:
baseline_results = []

for model_name, clf in models.items():
    pipe = Pipeline([("clf", clf)])

    scores = cross_validate(pipe, X, y, cv=CV, scoring=SCORING, n_jobs=-1)

    baseline_results.append({
        "experiment": f"Baseline | {model_name}",
        "acc_avg": float(scores["test_accuracy"].mean()),
        "balanced_acc_avg": float(scores["test_balanced_accuracy"].mean()),
        "f1_avg": float(scores["test_f1"].mean()),
        "roc_auc_avg": float(scores["test_roc_auc"].mean()),
    })

baseline_df = pd.DataFrame(baseline_results).sort_values("f1_avg", ascending=False)
baseline_df


,experiment,acc_avg,balanced_acc_avg,f1_avg,roc_auc_avg
6,Baseline | Stacking,0.915759,0.910729,0.877721,0.969013
0,Baseline | RandomForest,0.912480,0.899120,0.869116,0.973920
7,Baseline | Voting,0.909307,0.896681,0.865505,0.969681
4,Baseline | SVM,0.906029,0.894242,0.861270,0.954988
5,Baseline | Logistic Regression,0.902803,0.887157,0.851573,0.951440
2,Baseline | XGBoost,0.896298,0.882157,0.843474,0.964001
3,Baseline | Naive Bayes,0.863670,0.848354,0.801030,0.925958
1,Baseline | AdaBoost,0.850714,0.833769,0.778611,0.833769


<a id="model-on-dim-red"></a>
## Model Training on Dimensionality-Reduced Feature Spaces
---

In [11]:
embeddings = {
    "PCA(3)": joblib.load("../outputs/dim_red/embeddings/X_pca.pkl"),
    "LDA": joblib.load("../outputs/dim_red/embeddings/X_lda.pkl"),
    "t-SNE (3)": joblib.load("../outputs/dim_red/embeddings/X_tsne.pkl"),
    "UMAP (3)": joblib.load("../outputs/dim_red/embeddings/X_umap.pkl"),
    "mix": joblib.load("../outputs/dim_red/embeddings/X_mix.pkl"),
    # "UMAP supervised (3)": joblib.load("../outputs/dim_red/embeddings/X_umap_supervised.pkl"),
    "UMAP semi-supervised (3)": joblib.load("../outputs/dim_red/embeddings/X_umap_semisupervised.pkl"),
    "XGBoost + UMAP (3)": joblib.load("../outputs/dim_red/embeddings/X_umap_xgb.pkl"),
}

In [13]:
embedding_results = []

for emb_name, X_emb in embeddings.items():
    X_emb_eval = X_emb.to_numpy()

    for model_name, clf in models.items():
        pipe = Pipeline([("clf", clf)])

        scores = cross_validate(pipe, X_emb_eval, y, cv=CV, scoring=SCORING, n_jobs=-1)

        embedding_results.append({
            "experiment": f"{emb_name} + {model_name}",
            "acc_avg": float(scores["test_accuracy"].mean()),
            "balanced_acc_avg": float(scores["test_balanced_accuracy"].mean()),
            "f1_avg": float(scores["test_f1"].mean()),
            "roc_auc_avg": float(scores["test_roc_auc"].mean()),
        })

embedding_df = pd.DataFrame(embedding_results).sort_values("f1_avg", ascending=False)
embedding_df


,experiment,acc_avg,balanced_acc_avg,f1_avg,roc_auc_avg
44,UMAP semi-supervised (3) + SVM,0.993496,0.995061,0.990698,0.993426
40,UMAP semi-supervised (3) + RandomForest,0.987044,0.987860,0.981380,0.998943
46,UMAP semi-supervised (3) + Stacking,0.987044,0.985537,0.980947,0.995174
42,UMAP semi-supervised (3) + XGBoost,0.987044,0.985537,0.980947,0.991902
47,UMAP semi-supervised (3) + Voting,0.987044,0.985537,0.980947,0.995894
43,UMAP semi-supervised (3) + Naive Bayes,0.983765,0.980775,0.976064,0.992941
45,UMAP semi-supervised (3) + Logistic Regression,0.983765,0.980775,0.976064,0.996283
41,UMAP semi-supervised (3) + AdaBoost,0.980487,0.978336,0.971645,0.978336
48,XGBoost + UMAP (3) + RandomForest,0.896404,0.884724,0.849481,0.948467
54,XGBoost + UMAP (3) + Stacking,0.889952,0.884431,0.845452,0.945894


<a id="conclusions"></a>
## Conclusions
---

In [14]:
all_results = pd.concat([baseline_df, embedding_df], ignore_index=True)
all_results_sorted = all_results.sort_values("roc_auc_avg", ascending=False).reset_index(drop=True)
all_results_sorted


,experiment,acc_avg,balanced_acc_avg,f1_avg,roc_auc_avg
0,UMAP semi-supervised (3) + RandomForest,0.987044,0.987860,0.981380,0.998943
1,UMAP semi-supervised (3) + Logistic Regression,0.983765,0.980775,0.976064,0.996283
2,UMAP semi-supervised (3) + Voting,0.987044,0.985537,0.980947,0.995894
3,UMAP semi-supervised (3) + Stacking,0.987044,0.985537,0.980947,0.995174
4,UMAP semi-supervised (3) + SVM,0.993496,0.995061,0.990698,0.993426
...,...,...,...,...,...
59,XGBoost + UMAP (3) + Naive Bayes,0.876785,0.832985,0.785468,0.823659
60,UMAP (3) + AdaBoost,0.834215,0.812282,0.751370,0.812282
61,mix + AdaBoost,0.831200,0.800735,0.742385,0.800735
62,PCA(3) + AdaBoost,0.814754,0.795142,0.730011,0.795142


In [10]:
OUTPUT_DIR = "../outputs/model_training"
os.makedirs(OUTPUT_DIR, exist_ok=True)

out_path = os.path.join(OUTPUT_DIR, "embedding_vs_baseline_cv_results.csv")
all_results_sorted.to_csv(out_path, index=False)
print("Saved:", out_path)


Saved: ../outputs/model_training\embedding_vs_baseline_cv_results.csv
